RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/user/Artificial Intelligence/project_1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Data Ingestion
## Read all pdfs inside a directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob('*.pdf'))
    
    print(f"Found {len(pdf_files)} pdf files")
    
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try: 
            loader = PyMuPDFLoader(pdf_file)
            documents = loader.load()
        
            for doc in documents:
                doc.metadata['source'] = pdf_file.name
                doc.metadata['file_type']= 'pdf'
        
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} documents")
        
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
        
    print(f"Loaded a total of {len(all_documents)} documents")
    return all_documents

In [3]:
all_pdf_documents = process_all_pdfs(pdf_directory='../data/pdf')

Found 3 pdf files
Processing Untitled document.pdf
 Loaded 1 documents
Processing Hildesheim SE LoM.pdf
 Loaded 2 documents
Processing Pforzheim DBM LoM.pdf
 Loaded 2 documents
Loaded a total of 5 documents


In [4]:
### Text Splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=["\n\n", "\n", " ", ""])
    splits = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(splits)} chunks")
    
    
    if splits:
        print(f"\n Example Chunk:")
        print(f"Content: {splits[0].page_content[:chunk_overlap]}...")
        print(f"Metadata: {splits[0].metadata}")
    
    return splits

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 5 documents into 14 chunks

 Example Chunk:
Content: Machine Learning-Enhanced Bus Identification and Route Finder 
The Machine Learning-Enhanced Bus Ide...
Metadata: {'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'Untitled document.pdf', 'file_path': '../data/pdf/Untitled document.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Untitled document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'Untitled document.pdf', 'file_path': '../data/pdf/Untitled document.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Untitled document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}, page_content='Machine Learning-Enhanced Bus Identification and Route Finder \nThe Machine Learning-Enhanced Bus Identification and Route Finder system is designed to \nmodernize and optimize university transportation management by integrating intelligent \nautomation with user-friendly digital solutions. Traditional bus management systems rely heavily \non manual processes, which often lead to inefficiencies, delays, and lack of real-time \ninformation access. This project addresses these limitations by introducing a smart, \ntechnology-driven system that improves both operational efficiency 

Embedding and Vector DB Store

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        try:
            print("Loading model...")   # 👈 add this
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded!")      # 👈 add this
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} documents")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

manager = EmbeddingManager()
embeddings = manager.generate_embeddings(["Hello", "AI"])

Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10154.98it/s]


Model loaded!
Generating embeddings for 2 documents


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.16it/s]

Generated embeddings with shape: (2, 384)


In [10]:
### Vector Store

class VectorStore:
    """"MANAGE VECTOR STORE"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vectorstore"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize Chroma DB client and collection"""
        
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF documents embedding for RAG"
                    }
                )
                
            print(f"Created collection: {self.collection_name}")
            print(f"Existing Documents in Collection: {self.collection.count()}")
                
        except Exception as e:
            print(f"Error creating persist directory: {e}")
            raise
        
        
    def add_documents(self, documents: List[Any], embeddings = np.ndarray):
        
        
        if(len(documents) != len(embeddings)):
            raise ValueError("Number of documents and embeddings do not match")
        
        print(f"Adding {len(documents)} documents to vector store")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
       
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
           doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
           ids.append(doc_id)
           metadata = dict(doc.metadata)
           metadata["doc_index"] = i
           metadata['content_length'] = len(doc.page_content)
           
           metadatas.append(metadata)
           documents_text.append(doc.page_content)
           embeddings_list.append(embedding.tolist())
           
        try: 
            self.collection.add(
                documents=documents_text,
                embeddings=embeddings_list,
                metadatas=metadatas,
                ids=ids
            )
            
            print(f"Added {len(documents)} documents to vector store")
            print(f"Existing Documents in Collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorStore = VectorStore()
vectorStore
# vectorStore.add_documents(all_pdf_documents, embeddings)
       
    

Created collection: pdf_documents
Existing Documents in Collection: 42


In [11]:
texts = [doc.page_content for doc in chunks]


embeddings = manager.generate_embeddings(texts)


vectorStore.add_documents(chunks, embeddings)

Generating embeddings for 14 documents


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.82it/s]

Generated embeddings with shape: (14, 384)
Adding 14 documents to vector store
Added 14 documents to vector store
Existing Documents in Collection: 56


In [13]:
class RagRetreiver:
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def get_relevant_documents(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print("Retrieving documents for query:", query)
        print(f"Top-K: {top_k}, Score Threshold: {score_threshold}")
        
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                print(distances)
                
                for i, (doc, metadata, distance, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                    
                    similarity_score = distance
                    
                    print(f"Similarity Score: {similarity_score}")
                    print(f"SCORE THRESHOLD: {score_threshold}")
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "page_content": doc,
                            "metadata": metadata,
                            "distance": distance,
                            "doc_id": doc_id,
                            "rank": i+1
                        })
                        
                print(f"Retrieved {len(retrieved_docs)} documents")
                return retrieved_docs
            else:
                return retrieved_docs
                
            
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise

In [14]:
retriever = RagRetreiver(vectorStore, manager)
retriever.get_relevant_documents("Where is Hildesheim")

Retrieving documents for query: Where is Hildesheim
Top-K: 5, Score Threshold: 0.0
Generating embeddings for 1 documents


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.97it/s]

Generated embeddings with shape: (1, 384)
[1.0960149765014648, 1.0960149765014648, 1.0960149765014648, 1.0960149765014648, 1.4029245376586914]
Similarity Score: 1.0960149765014648
SCORE THRESHOLD: 0.0
Similarity Score: 1.0960149765014648
SCORE THRESHOLD: 0.0
Similarity Score: 1.0960149765014648
SCORE THRESHOLD: 0.0
Similarity Score: 1.0960149765014648
SCORE THRESHOLD: 0.0
Similarity Score: 1.4029245376586914
SCORE THRESHOLD: 0.0
Retrieved 5 documents


[{'page_content': 'because of its focus on practical application and modern development approaches. I am \nespecially interested in gaining knowledge about software design patterns, development \nmethodologies, and system optimization techniques. I believe that this program will help me \nconnect my existing practical skills with strong theoretical knowledge, which is essential for \nbecoming a well-rounded software engineer. \nGermany is known for its high-quality education system and strong emphasis on research \nand innovation. Studying at University of Hildesheim will give me the opportunity to learn in an',
  'metadata': {'trapped': '',
   'file_path': '../data/pdf/Hildesheim SE LoM.pdf',
   'source': 'Hildesheim SE LoM.pdf',
   'title': 'Hildesheim SE LoM',
   'keywords': '',
   'creationDate': '',
   'moddate': '',
   'total_pages': 2,
   'producer': 'Skia/PDF m149 Google Docs Renderer',
   'format': 'PDF 1.4',
   'file_type': 'pdf',
   'content_length': 592,
   'author': '',
  

Integration VectorDB Context pipeline with LLM Output

In [15]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
import os

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=groq_api_key, model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k=3):

    results = retriever.get_relevant_documents(query, top_k=top_k)

    print(results)
    print(type(results[0]))

    if not results:
        return "No relevant documents found"

    context = "\n\n".join([doc['page_content'] for doc in results])

    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content


In [16]:
answer = rag_simple("what i do?", retriever, llm)
print(answer)

Retrieving documents for query: what i do?
Top-K: 3, Score Threshold: 0.0
Generating embeddings for 1 documents


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.17it/s]

Generated embeddings with shape: (1, 384)
[1.64095938205719, 1.64095938205719, 1.64095938205719]
Similarity Score: 1.64095938205719
SCORE THRESHOLD: 0.0
Similarity Score: 1.64095938205719
SCORE THRESHOLD: 0.0
Similarity Score: 1.64095938205719
SCORE THRESHOLD: 0.0
Retrieved 3 documents
[{'page_content': 'concepts and practical aspects of computing systems. I also worked on several academic \nprojects, including a Library Management System, a Smart Bus Finder application, and a \nBlood Donation platform. Through these projects, I gained hands-on experience in system \ndesign, data handling, and application development. More importantly, I learned how to \napproach problems step by step and develop solutions that are both practical and user-friendly. \nAlong with my academic experience, I have developed strong professional skills through my \ncurrent role as a Lead Flutter Developer at Technoverse. In this position, I design and \ndevelop scalable mobile applications that support real-ti

I design and develop scalable mobile applications, specifically as a Lead Flutter Developer at Technoverse, with a focus on system design, data handling, and application development.


In [21]:
def rag_advance(query, retriever, llm, top_k=3, min_score=0.1, return_context=False):

    results = retriever.get_relevant_documents(query, top_k=top_k, score_threshold=min_score)


    if not results:
        return "No relevant documents found"

    context = "\n\n".join([doc['page_content'] for doc in results])
    
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['distance'],
        'preview': doc['page_content'][:20]
    } for doc in results]
    
    confidence = max([doc['distance'] for doc in results])


    prompt = f"""
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt.format(context=context, query=query))
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if(return_context):
        output['context'] = context

    return output



In [24]:
answer = rag_simple("what i have done?", retriever, llm)
print(answer)

Retrieving documents for query: what i have done?
Top-K: 3, Score Threshold: 0.0
Generating embeddings for 1 documents


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.77it/s]

Generated embeddings with shape: (1, 384)
[1.727393388748169, 1.727393388748169, 1.727393388748169]
Similarity Score: 1.727393388748169
SCORE THRESHOLD: 0.0
Similarity Score: 1.727393388748169
SCORE THRESHOLD: 0.0
Similarity Score: 1.727393388748169
SCORE THRESHOLD: 0.0
Retrieved 3 documents
[{'page_content': 'concepts and practical aspects of computing systems. I also worked on several academic \nprojects, including a Library Management System, a Smart Bus Finder application, and a \nBlood Donation platform. Through these projects, I gained hands-on experience in system \ndesign, data handling, and application development. More importantly, I learned how to \napproach problems step by step and develop solutions that are both practical and user-friendly. \nAlong with my academic experience, I have developed strong professional skills through my \ncurrent role as a Lead Flutter Developer at Technoverse. In this position, I design and \ndevelop scalable mobile applications that support r

You have worked on several academic projects (Library Management System, Smart Bus Finder, Blood Donation platform) and developed scalable mobile applications as a Lead Flutter Developer at Technoverse, including systems for incident reporting and responder coordination.
